In [2]:
import os
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')


# 2. Điền thông tin Kaggle API của bạn vào đây
# (Vào Kaggle -> Settings -> Creatxe New Token để lấy username và key)
os.environ['KAGGLE_USERNAME'] = "huyphngquc"
os.environ['KAGGLE_KEY'] = "d2051dddb134d7809d29527c977cab58"

# Tạo thư mục trên Drive để lưu dataset cố định (nếu chưa có)
DRIVE_DATASET_DIR = '/content/drive/MyDrive/Thesis_ViVQA_Data'
os.makedirs(DRIVE_DATASET_DIR, exist_ok=True)

Mounted at /content/drive


In [3]:
import os
import time

LOCAL_DATA_DIR = '/content/vivqa_dataset'
ZIP_PATH_ON_DRIVE = f'{DRIVE_DATASET_DIR}/vivqa-1.zip' # Đảm bảo biến này đúng
ZIP_PATH_LOCAL = '/content/vivqa-1.zip'

start_time = time.time()

# 1. Xóa rác cũ nếu có (Quan trọng để tránh giải nén đè lên file lỗi)
!rm -rf {LOCAL_DATA_DIR}
!rm -f {ZIP_PATH_LOCAL}

# 2. Copy bằng lệnh Linux (Trâu hơn shutil)
print("Đang copy từ Drive sang Local...")
!cp "{ZIP_PATH_ON_DRIVE}" "{ZIP_PATH_LOCAL}"

# 3. Giải nén và TỰ ĐỘNG SỬA LỖI (Dùng option -FF nếu cần)
print("Đang giải nén...")
os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

# Thử giải nén bình thường trước
result = !unzip -q -n {ZIP_PATH_LOCAL} -d {LOCAL_DATA_DIR}

# Kiểm tra nếu vẫn còn báo lỗi "bad zipfile offset"
if any("bad zipfile offset" in line for line in result):
    print("⚠️ File zip có vẻ bị lỗi offset, đang thử fix...")
    # Lệnh fix file zip của linux
    !zip -F {ZIP_PATH_LOCAL} --out {ZIP_PATH_LOCAL}_fixed.zip
    !unzip -q -n {ZIP_PATH_LOCAL}_fixed.zip -d {LOCAL_DATA_DIR}

# 4. Dọn dẹp
if os.path.exists(ZIP_PATH_LOCAL): os.remove(ZIP_PATH_LOCAL)

print(f"✅ Xong! Kiểm tra thử: {len(os.listdir(LOCAL_DATA_DIR))} folders/files.")

✅ Xong! Kiểm tra thử: 13 folders/files.


In [4]:
# 4. Kiểm tra và in ra cấu trúc thực tế
print("\n--- KIỂM TRA CẤU TRÚC DATA ---")
if os.path.exists(LOCAL_DATA_DIR):
    # Liệt kê tất cả file và folder (kể cả trong folder con)
    all_files = []
    for root, dirs, files in os.walk(LOCAL_DATA_DIR):
        for name in files:
            all_files.append(os.path.join(root, name))
    
    print(f"📍 Path cơ sở: {LOCAL_DATA_DIR}")
    print(f"📦 Tổng số lượng file tìm thấy: {len(all_files)}")
    
    if len(all_files) > 0:
        print("📄 5 file đầu tiên để check path:")
        for f in all_files[:5]:
            print(f"   -> {f}")
    else:
        print("❌ CẢNH BÁO: Thư mục tồn tại nhưng không có file nào bên trong (do giải nén lỗi).")
else:
    print("❌ LỖI: Thư mục LOCAL_DATA_DIR không tồn tại.")


--- KIỂM TRA CẤU TRÚC DATA ---
📍 Path cơ sở: /content/vivqa_dataset
📦 Tổng số lượng file tìm thấy: 11720
📄 5 file đầu tiên để check path:
   -> /content/vivqa_dataset/answer_space.txt
   -> /content/vivqa_dataset/dino_final_results.csv
   -> /content/vivqa_dataset/yolo_final_results.csv
   -> /content/vivqa_dataset/vqa_classified_answers.csv
   -> /content/vivqa_dataset/merged_train.csv


In [5]:
!pip install -q transformers timm accelerate bitsandbytes scikit-learn underthesea

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 135.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 94.9 MB/s eta 0:00:00


In [6]:
import warnings, logging
warnings.filterwarnings("ignore", category=UserWarning)
logging.getLogger("torch.utils.data.dataloader").setLevel(logging.ERROR)

In [ ]:
import torch

# ==========================================
# BOX CONFIG — Chỉnh tham số tại đây
# Tối ưu cho GPU A100 40GB
# ==========================================
BOX_CONFIG = {
    "train_csv": "/content/vivqa_dataset/ViVQA-main/ViVQA-main/train.csv",
    "train_img_dir": "/content/vivqa_dataset/train",
    "train_box_csv": "/content/vivqa_dataset/merged_train.csv",

    "test_csv": "/content/vivqa_dataset/ViVQA-main/ViVQA-main/test.csv",
    "test_img_dir": "/content/vivqa_dataset/test",

    "save_path": "/content/drive/MyDrive/AI_Models/vivqa_box.pth",

    "blip_model": "Salesforce/blip2-opt-2.7b",
    "text_model": "vinai/phobert-base",

    # A100 40GB: tăng batch_size 32→64, num_workers 0→4
    "batch_size": 64,
    "epochs": 25,
    "lr": 5e-5,
    "weight_decay": 0.05,

    "lambda_bbox": 2.0,

    "type_embed_dim": 64,
    "device": "cuda" if torch.cuda.is_available() else "cpu",

    "max_length": 50,
    "patience": 5,
    "delta": 0.001,
    "num_workers": 4,

    # Trọng số loss theo q_type (tăng = ép model học kỹ hơn loại đó)
    "type_loss_weights": {0: 4.0, 1: 0.5, 2: 4.0, 3: 4.0},
}


In [ ]:
# ==========================================
# V4 CONFIG — Chỉnh tham số tại đây
# Tối ưu cho GPU A100 40GB
# ==========================================
V4_CONFIG = {
    # File train gốc (tự tách 90/10 thành Train + Val nội bộ)
    "train_csv": "/content/vivqa_dataset/ViVQA-main/ViVQA-main/train.csv",
    "train_img_dir": "/content/vivqa_dataset/train",

    # File test (hold-out, chỉ dùng cho inference)
    "test_csv": "/content/vivqa_dataset/ViVQA-main/ViVQA-main/test.csv",
    "test_img_dir": "/content/vivqa_dataset/test",

    # File mapping loại câu hỏi → đáp án (tạo Hierarchical Mask)
    "type_mapping_csv": "/content/vivqa_dataset/answer_type_mapping.csv",

    "save_path": "/content/drive/MyDrive/AI_Models/vivqa_v4.pth",

    "blip_model": "Salesforce/blip2-opt-2.7b",
    "text_model": "vinai/phobert-base",

    # A100 40GB: tăng batch_size 32→64, num_workers 0→4
    "batch_size": 64,
    "epochs": 25,
    "lr": 5e-5,
    "weight_decay": 0.05,
    "lambda_type": 0.5,

    "device": "cuda" if torch.cuda.is_available() else "cpu",

    "max_length": 50,
    "patience": 5,
    "num_workers": 4,

    # Trọng số loss theo q_type (boost type 1 vì khó nhất)
    "type_loss_weights": {0: 1.0, 1: 3.0, 2: 1.0, 3: 1.0},
}


# ViVQA — Main Training Notebook

Notebook này điều phối quá trình huấn luyện và inference cho **hai mô hình**:

| Mô hình | Thư mục | Mô tả |
|---------|---------|-------|
| **Box** | `Box/` | Đa nhiệm: phân loại câu trả lời + dự đoán Bounding Box |
| **V4**  | `V4/`  | Hierarchical VQA với Type Mask theo loại câu hỏi |

**Cấu trúc thư mục:**
```
DHM/
├── main.ipynb          ← Notebook điều phối (file này)
├── utils/
│   ├── preprocessing.py   ← preprocess_vietnamese, find_image_path
│   ├── early_stopping.py  ← EarlyStopping
│   └── fusion.py          ← CrossAttentionFusion
├── Box/
│   ├── dataset.py         ← ViVQADataset, ViVQAMultiTaskInferenceDataset
│   ├── model.py           ← PhoBERT_BLIP2_EfficientNet_MultiTask
│   ├── trainer.py         ← Vòng lặp huấn luyện
│   └── inference.py       ← Chạy inference và xuất CSV
└── V4/
    ├── dataset.py         ← ViVQADataset, ViVQAInferenceDataset
    ├── model.py           ← PhoBERT_BLIP2_VQA_Hierarchical, create_answer_type_mask
    ├── trainer.py         ← Vòng lặp huấn luyện
    └── inference.py       ← Chạy inference và xuất CSV
```

In [9]:
import sys
import os

# Đảm bảo thư mục DHM/ nằm trong Python path để import được các module
notebook_dir = os.path.dirname(os.path.abspath("main.ipynb"))
if notebook_dir not in sys.path:
    sys.path.insert(0, notebook_dir)

## Mô hình 1 — Box (Đa nhiệm: VQA + Bounding Box)

**Mô tả:** Dùng BLIP-2 (Q-Former) + EfficientNet-B7 + PhoBERT kết hợp qua CrossAttention.  
Hai đầu ra: phân loại câu trả lời và hồi quy tọa độ Bounding Box (chuẩn hóa [0, 1]).  
Có thể chỉnh `CONFIG` trong `Box/config.py`.

In [ ]:
!git clone https://github.com/ThinhLe09/testMLmodel
%cd testMLmodel

Cloning into 'testMLmodel'...
remote: Enumerating objects: 40, done.
remote: Counting objects: 100% (40/40), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 40 (delta 14), reused 3 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (40/40), 78.30 KiB | 1.82 MiB/s, done.
Resolving deltas: 100% (14/14), done.


In [17]:

from Box.trainer import train as train_box
train_box(BOX_CONFIG)

/content/testMLmodel


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

...

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x79dd6ddccfe0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x79dd6ddccfe0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
self._shutdown_workers()    
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

Epoch 6 | Train Acc: 0.6161 | Val Acc: 0.5975
    --- Per Q-Type Answer Accuracy ---
    Type 0 (w=4.0): Train=0.6758 | Val=0.6712 <<<
    Type 1 (w=0.5): Train=0.3784 | Val=0.3716
    Type 2 (w=4.0): Train=0.6253 | Val=0.6224 <<<
    Type 3 (w=4.0): Train=0.6530 | Val=0.5869 <<<


Epoch 7:   0%|          | 0/375 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x79dd6ddccfe0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x79dd6ddccfe0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

KeyboardInterrupt: 

In [ ]:
from Box.inference import run_inference_and_save as box_inference
box_inference(BOX_CONFIG)

In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()

## Mô hình 2 — V4 (Hierarchical VQA với Type Mask)

**Mô tả:** Dùng BLIP-2 Vision Encoder (raw features, dim=1408) + PhoBERT kết hợp qua CrossAttention.  
Hai đầu ra: dự đoán loại câu hỏi (type classifier) + phân loại câu trả lời với mask theo từng loại câu hỏi.  
Tập train được tự chia thành Train/Val (90/10) theo stratify `type`.  
Có thể chỉnh `CONFIG` trong `V4/config.py`.

In [ ]:
from V4.trainer import train as train_v4
train_v4(V4_CONFIG)

In [ ]:
from V4.inference import run_inference_and_save as v4_inference
v4_inference(V4_CONFIG)

## Mô hình 3 — Ensemble (Routing: V4 + Box)

**Mô tả pipeline:**
1. **V4** dự đoán `q_type` + câu trả lời cho từng mẫu.
2. **Box** nhận `q_type` từ V4 (không dùng ground truth) → dự đoán câu trả lời + bounding box.
3. **Routing:** `q_type == 1` → dùng V4 answer; còn lại → dùng Box answer.

> Chạy sau khi cả Box và V4 đã train xong.

In [ ]:
from Ensemble.inference import run_ensemble_inference

run_ensemble_inference(BOX_CONFIG, V4_CONFIG, output_file="ket_qua_ensemble.csv", batch_size=32)

---
## Inference từ Checkpoint có sẵn

Dùng khi đã có file `.pth` và **không muốn train lại**.  
Điền đường dẫn checkpoint rồi chạy từng cell độc lập.

### Box — Inference từ checkpoint

In [ ]:
from Box.inference import run_inference_and_save as box_inference

BOX_CHECKPOINT_PATH = "/content/drive/MyDrive/AI_Models/vivqa_box.pth"
BOX_OUTPUT_FILE     = "ket_qua_box.csv"

box_inference({**BOX_CONFIG, "save_path": BOX_CHECKPOINT_PATH}, output_file=BOX_OUTPUT_FILE)

### V4 — Inference từ checkpoint

In [ ]:
from V4.inference import run_inference_and_save as v4_inference

V4_CHECKPOINT_PATH = "/content/drive/MyDrive/AI_Models/vivqa_v4.pth"
V4_OUTPUT_FILE     = "ket_qua_v4.csv"

v4_inference({**V4_CONFIG, "save_path": V4_CHECKPOINT_PATH}, output_file=V4_OUTPUT_FILE)

### Ensemble — Inference từ checkpoint

In [ ]:
from Ensemble.inference import run_ensemble_inference

BOX_CHECKPOINT_PATH = "/content/drive/MyDrive/AI_Models/vivqa_box.pth"
V4_CHECKPOINT_PATH  = "/content/drive/MyDrive/AI_Models/vivqa_v4.pth"
ENS_OUTPUT_FILE     = "ket_qua_ensemble.csv"

run_ensemble_inference(
    box_config={**BOX_CONFIG, "save_path": BOX_CHECKPOINT_PATH},
    v4_config={**V4_CONFIG,  "save_path": V4_CHECKPOINT_PATH},
    output_file=ENS_OUTPUT_FILE,
    batch_size=32,
)